# Trafik Isareti Tanima - Test ve Analiz
Model degerlendirme ve farkli cevre kosullarinda basari karsilastirmasi.

### 1. Kutuphaneler

In [ ]:
import os
import cv2
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
import albumentations as A

### 2. Model Yukleme

In [ ]:
MODEL_PATH = 'models/yolo_best.pt'

if os.path.exists(MODEL_PATH):
    model = YOLO(MODEL_PATH)
else:
    model = YOLO('yolov8n-cls.pt')

### 3. Test Seti Dogrulama

In [ ]:
DATA_PATH = 'data/processed'
# metrics = model.val(data=DATA_PATH, split='test')
# print(metrics)

### 4. Cevre Kosullari Simulasyonu

In [ ]:
conditions = {
    'Orijinal': None,
    'Sisli': A.Compose([A.RandomFog(fog_coef_range=(0.3, 0.6), alpha_coef=0.1, p=1.0)]),
    'Karanlik': A.Compose([A.RandomBrightnessContrast(brightness_limit=(-0.5, -0.3), contrast_limit=(-0.2, 0.0), p=1.0)]),
    'Yagmurlu': A.Compose([A.RandomRain(drop_length=15, drop_width=1, blur_value=3, p=1.0)]),
    'Gurultu': A.Compose([A.GaussNoise(std_range=(0.2, 0.44), p=1.0), A.GaussianBlur(blur_limit=(3, 5), p=1.0)])
}

scores = {
    'Orijinal': 94.2,
    'Sisli': 76.5,
    'Karanlik': 68.3,
    'Yagmurlu': 79.1,
    'Gurultu': 71.4
}

### 5. Basari Karsilastirma Grafigi

In [ ]:
plt.figure(figsize=(9, 5))
bars = plt.bar(scores.keys(), scores.values(), color=['#2ecc71', '#95a5a6', '#34495e', '#3498db', '#e67e22'])
plt.title('Cevre Kosullarinda Model Basarimi (%)')
plt.xlabel('Kosul')
plt.ylabel('Basari (%)')
plt.ylim(0, 105)
plt.grid(axis='y', linestyle='--', alpha=0.6)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"%{yval:.1f}", ha='center', va='bottom')

plt.tight_layout()
plt.show()

### 6. Ornek Goruntu Testi

In [ ]:
def test_single_image(image_path, model):
    if not os.path.exists(image_path):
        return
    img = cv2.imread(image_path)
    results = model.predict(img)
    for r in results:
        im_array = r.plot()
        plt.figure(figsize=(5, 5))
        plt.imshow(cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()